In [1]:
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as nc4
import glob
import datetime as dt
import pickle as pkl
from scipy.interpolate import interp1d, RegularGridInterpolator
from matplotlib.ticker import ScalarFormatter
import os
import jdcal
import sim_name_remap as map
import plotter_config as pc

In [2]:

def get_ACCLIP_merge_data(OBSROOT, FLT_DATES, var_str):
    
    OBSvar = []

    if var_str != '':
        for FLT_DATE in FLT_DATES:
        
            AIRFILE = glob.glob(OBSROOT + '*' + FLT_DATE + '*.ict')[0]     
            AIRheaders = np.loadtxt(AIRFILE, delimiter=',', usecols = (0), dtype='str')
            AIRdata = np.loadtxt(AIRFILE, delimiter=',', skiprows=int(AIRheaders[0]))  # Read the file
            OBSvar.extend(AIRdata[:,int(list(AIRheaders).index(var_str))-11])
        
    return np.asarray(OBSvar)
    

In [3]:

def get_air_jd(FLT_DATES, OBSROOT):

    air_jd = []
    
    for FLT in FLT_DATES:
        
        curr_sec_array = get_ACCLIP_merge_data(OBSROOT, [FLT], 'Mid_Time_UTC')
        
        for pt in range(len(curr_sec_array)):
            curr_sec = curr_sec_array[pt]
            curr_jd = np.sum(jdcal.gcal2jd(int(FLT[0:4]), int(FLT[4:6]), int(FLT[6:]))) + curr_sec/86400.
            air_jd.append(curr_jd)

    return air_jd


In [4]:

AIRPLANE = 'GV'
FLT_DATES = ['20220819']
sim = '2022_E5-strong'  
var_lab = 'CO'
curtain_prs_grid = np.arange(0,1001,25)
SAVEDIR = '/glade/u/home/wsmith/NudgeComparison_scripts/vars/'

date0 = '2022-08-18'  # These are used just to bookend the event and not read 40,000 files
date1 = '2022-08-20'


In [5]:

# Read the observations onto nice time series in JD, lon, lat, and prs

if AIRPLANE == 'GV':
    
    GV_OBSROOT = '/glade/work/wsmith/ACCLIP_GV_R1_60SEC_Merge/'    
    air_lon = get_ACCLIP_merge_data(GV_OBSROOT, FLT_DATES, 'GGLON') 
    air_lat = get_ACCLIP_merge_data(GV_OBSROOT, FLT_DATES, 'GGLAT') 
    air_prs = get_ACCLIP_merge_data(GV_OBSROOT, FLT_DATES, 'PSXC') 
    air_alt = get_ACCLIP_merge_data(GV_OBSROOT, FLT_DATES, 'GGALT')*1.e-3
    air_jd = get_air_jd(FLT_DATES, GV_OBSROOT)

if AIRPLANE == 'WB57':

    WB_OBSROOT = '/glade/work/wsmith/ACCLIP_WB57_R1_60SEC_Merge/'
    air_lon = get_ACCLIP_merge_data(WB_OBSROOT, FLT_DATES, 'G_LONG_MMS') 
    air_lat = get_ACCLIP_merge_data(WB_OBSROOT, FLT_DATES, 'G_LAT_MMS') 
    air_prs = get_ACCLIP_merge_data(WB_OBSROOT, FLT_DATES, 'P_MMS') 
    air_alt = get_ACCLIP_merge_data(WB_OBSROOT, FLT_DATES, 'G_ALT_MMS')*1.e-3
    air_jd = get_air_jd(FLT_DATES, WB_OBSROOT)    


In [6]:

# Define the subset of model output files we care about
FILES = sorted(glob.glob(map.remap[sim]))
FILES0 = [item for item in FILES if date0 in item][0]
find0 = list(FILES).index(FILES0)
FILES1 = [item for item in FILES if date1 in item][0]
find1 = list(FILES).index(FILES1)
FILES = FILES[find0:find1+1]

if var_lab == 'SO4': 
    var_strs = ['so4_a1','so4_a2','so4_a3']
else: var_strs = [var_lab]

# Read the model info, note that we need to just use the nearest point in time because of the required pressure correction

CESM_file0 = nc4.Dataset(FILES[0])
CESM_lon = CESM_file0.variables['lon'][:]
CESM_lat = CESM_file0.variables['lat'][:]
CESM_lev = CESM_file0.variables['lev'][:]
hyam = CESM_file0.variables['hyam'][:]
hybm = CESM_file0.variables['hybm'][:]
CESM_date = CESM_file0.variables['date'][:]
CESM_datesec = CESM_file0.variables['datesec'][:]
CESM_PS = CESM_file0.variables['PS'][:]

CESM_var = CESM_file0.variables[var_strs[0]][:]*pc.var_info[var_strs[0]]['mod']   # We apply each mod individually to scale for the mass ERICA could have sampled
for sub_v in range(len(var_strs)-1):  # In case there is more than one variable
    sub_var = CESM_file0.variables[var_strs[sub_v+1]][:]*pc.var_info[var_strs[sub_v+1]]['mod']
    CESM_var = CESM_var + sub_var

for f in range(len(FILES)-1):  # Loop over the rest of the files
    print(f)
    CESM_file = nc4.Dataset(FILES[f+1])
    CESM_PS = np.append(CESM_PS, CESM_file.variables['PS'][:], axis=0)
    CESM_date = np.append(CESM_date, CESM_file.variables['date'][:])
    CESM_datesec = np.append(CESM_datesec, CESM_file.variables['datesec'][:])

    curr_var = CESM_file.variables[var_strs[0]][:]*pc.var_info[var_strs[0]]['mod']
    for sub_v in range(len(var_strs)-1):  # In case there is more than one variable
        sub_var = CESM_file.variables[var_strs[sub_v+1]][:]*pc.var_info[var_strs[sub_v+1]]['mod']
        curr_var = curr_var + sub_var
    CESM_var = np.append(CESM_var, curr_var, axis=0)


0
1


In [7]:

# Convert the CESM dates to JD

CESM_jd = []
for pt in range(len(CESM_date)):
    curr_jd = np.sum(jdcal.gcal2jd(int(str(CESM_date[pt])[0:4]), int(str(CESM_date[pt])[4:6]), int(str(CESM_date[pt])[6:])))  + CESM_datesec[pt]/86400.
    CESM_jd.append(curr_jd)


In [8]:

f_CESM = RegularGridInterpolator((CESM_jd, CESM_lev, CESM_lat, CESM_lon), CESM_var, bounds_error=False, fill_value=-999)


In [9]:

# Loop over all the aircraft points and do a linear interpolation
# Note we look for the nearest model point because we have to calculate the levels manually at each point

CESM_var_interp = []
CESM_var_curtain = np.zeros(len(curtain_prs_grid))[None,:]

for pt in range(len(air_lon)):

    # These lines used to be used
    #curr_CESM_tind = np.argmin(np.abs(CESM_jd - GV_jd[pt]))
    #curr_CESM_lonind = np.argmin(np.abs(CESM_lon - GV_lon[pt]))
    #curr_CESM_latind = np.argmin(np.abs(CESM_lat - GV_lat[pt]))    
    #curr_CESM_PS = CESM_PS[curr_CESM_tind, curr_CESM_latind, curr_CESM_lonind]
    #curr_CESM_plev = (hyam*1.0e5 + hybm*curr_CESM_PS)/100.
    #curr_CESM_var = np.squeeze(CESM_var[curr_CESM_tind,:,curr_CESM_latind,curr_CESM_lonind])    
    #f_prs = interp1d(curr_CESM_plev, curr_CESM_var, bounds_error=False, fill_value=-999)
    #CESM_var_interp.append(f_prs(GV_prs[pt]))    

    # Instead of the above, we just use the RegularGridInterpolator and accept the limitation of the lev variable, we are mostly over the ocean anyways
    
    CESM_var_interp.append(f_CESM((air_jd[pt], air_prs[pt], air_lat[pt], air_lon[pt])))

    # Also save information for the full pressure grid so we can make cool curtain plots
    #curr_CESM_curtain = f_prs(curtain_prs_grid)[None,:]
    #CESM_var_curtain = np.append(CESM_var_curtain, curr_CESM_curtain, axis=0)
    
print(np.shape(CESM_var_curtain))

CESM_var_interp = np.asarray(CESM_var_interp)
#CESM_var_curtain = CESM_var_curtain[1:,:]

print(np.nanmin(CESM_var_interp), np.nanmax(CESM_var_interp))


(1, 41)
-999.0 208.38071107534196


In [10]:

#########  SAVE ALL THIS COOL INFO FOR LATER PLOTTING ELSEWHERE

with open(SAVEDIR + sim + '_' + AIRPLANE + '_' + var_lab + '_1D_interp.pkl','wb') as f:
    pkl.dump((air_lon, air_lat, air_prs, air_alt, air_jd, var_lab, CESM_var_interp), f)
